# Insurance Claims Classification Workbench

Portfolio source extract. See the repository README and PROVENANCE.md for methods, data requirements, original evidence and publication changes. Run from the notebooks/ directory.

In [ ]:
import sys
# Publication: install dependencies from ../requirements.txt first.


In [ ]:
import sys
# Publication: install dependencies from ../requirements.txt first.


In [ ]:
import sys
import pandas as pd
import numpy as np
import sklearn
import matplotlib
import seaborn
import lightgbm
import xgboost

print("✅ Python executable:", sys.executable)
print("📦 pandas:", pd.__version__)
print("📦 numpy:", np.__version__)
print("📦 scikit-learn:", sklearn.__version__)
print("📦 matplotlib:", matplotlib.__version__)
print("📦 seaborn:", seaborn.__version__)
print("📦 lightgbm:", lightgbm.__version__)
print("📦 xgboost:", xgboost.__version__)


In [ ]:
import pandas as pd

df = pd.read_csv("cleaned_claims_dataset.csv", low_memory=False)
print(df.shape)
df.head()


In [ ]:
# ==== 0) CONFIG ====
DATA_PATH = "cleaned_claims_dataset.csv"   # <-- put your final CSV path here
TARGET = "paid_status"

# ==== 1) LOAD ====
import pandas as pd, numpy as np
from lightgbm import early_stopping, log_evaluation

df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
assert TARGET in df.columns, f"Missing target '{TARGET}'"

# ==== 2) FEATURE SELECTION ====
# Drop free-text and leakage columns for v1 model
drop_cols = []
drop_cols += [c for c in df.columns if c.startswith("Remark")]                   # Remark1..Remark6
drop_cols += [c for c in df.columns if c.startswith("Paid") and c != TARGET]     # Paid1..Paid6
if "GeneralRemark" in df.columns: drop_cols.append("GeneralRemark")

X = df.drop(columns=drop_cols + [TARGET])
y = df[TARGET].astype(int)

# ==== 3) BASIC ENCODING ====
# Treat object columns as categorical (LightGBM handles pandas category natively)
cat_cols = [c for c in X.columns if X[c].dtype == "object"]
X[cat_cols] = X[cat_cols].fillna("None")
for c in cat_cols:
    X[c] = X[c].astype("category")

print(f"Features: {X.shape[1]} | Categorical: {len(cat_cols)}")

# ==== 4) SPLIT ====
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ==== 5) TRAIN (LightGBM) ====
import lightgbm as lgb
lgb_train = lgb.Dataset(X_train, label=y_train, categorical_feature=cat_cols, free_raw_data=False)
lgb_valid = lgb.Dataset(X_test,  label=y_test,  categorical_feature=cat_cols, free_raw_data=False)

params = {
    "objective": "binary",
    "metric": ["auc","binary_logloss"],
    "boosting_type": "gbdt",
    "num_leaves": 63,
    "learning_rate": 0.05,
    "feature_fraction": 0.9,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "verbose": -1,
    "n_jobs": -1,
}

model = lgb.train(
    params,
    lgb_train,
    valid_sets=[lgb_train, lgb_valid],
    valid_names=["train","valid"],
    num_boost_round=3000,
    callbacks=[
        early_stopping(stopping_rounds=100),
        log_evaluation(period=50),
    ],
)

# ==== 6) EVALUATE ====
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, precision_score, recall_score, confusion_matrix, classification_report

proba = model.predict(X_test, num_iteration=model.best_iteration)
pred  = (proba >= 0.5).astype(int)

print("Accuracy  :", accuracy_score(y_test, pred))
print("ROC-AUC   :", roc_auc_score(y_test, proba))
print("F1        :", f1_score(y_test, pred))
print("Precision :", precision_score(y_test, pred))
print("Recall    :", recall_score(y_test, pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, pred))
print("\nClassification Report:\n", classification_report(y_test, pred, digits=3))

# ==== 7) FEATURE IMPORTANCE ====
import matplotlib.pyplot as plt
import pandas as pd
imp = pd.Series(model.feature_importance(importance_type="gain"), index=X_train.columns).sort_values(ascending=False)
ax = imp.head(30).plot(kind="barh", figsize=(8,10))
ax.set_title("Top 30 Feature Importances (gain)")
plt.gca().invert_yaxis()
plt.show()

# ==== 8) SAVE ARTIFACTS ====
import joblib
joblib.dump(model, "paid_claim_predictor_lightgbm.pkl")
X_test.assign(paid_status_true=y_test, paid_status_pred=pred, paid_prob=proba).head(50).to_csv("sample_predictions.csv", index=False)
print("Saved model -> paid_claim_predictor_lightgbm.pkl")
print("Saved sample predictions -> sample_predictions.csv")


In [ ]:
leak_cols = []
leak_cols += [c for c in df.columns if c.startswith("Paid") and c != "paid_status"]
leak_cols += [c for c in df.columns if c.startswith("Remark")]
leak_cols += [c for c in df.columns if c.endswith("_Adj")]  # new
if "GeneralRemark" in df.columns: leak_cols.append("GeneralRemark")

X = df.drop(columns=leak_cols + ["paid_status"])


In [ ]:
df = pd.read_csv(DATA_PATH, low_memory=False)
# Make all non-numeric coding columns explicit strings
for c in df.columns:
    if df[c].dtype == 'object':
        df[c] = df[c].astype('string')


In [ ]:
import numpy as np
from sklearn.metrics import f1_score, precision_score, recall_score

best = max(
    ((t, f1_score(y_test, (proba>=t).astype(int)),
         precision_score(y_test, (proba>=t).astype(int)),
         recall_score(y_test, (proba>=t).astype(int)))
     for t in np.linspace(0.1, 0.9, 33)),
    key=lambda x: x[1]
)
print("Best threshold / F1 / Precision / Recall:", best)


In [ ]:
from sklearn.metrics import precision_recall_fscore_support
import numpy as np

for t in np.linspace(0.1, 0.9, 33):
    pred = (proba >= t).astype(int)
    precision, recall, f1, _ = precision_recall_fscore_support(y_test, pred, average=None, labels=[0, 1])
    
    # precision[0], recall[0], f1[0] correspond to class 0
    print(f"t={t:.2f} | class0 F1={f1[0]:.3f} P={precision[0]:.3f} R={recall[0]:.3f}")


In [ ]:
from sklearn.calibration import CalibratedClassifierCV
from lightgbm import LGBMClassifier

clf = LGBMClassifier(**{k:v for k,v in params.items() if k in [
    "objective","boosting_type","num_leaves","learning_rate",
    "feature_fraction","bagging_fraction","bagging_freq","n_jobs"
]})
clf.fit(X_train, y_train, categorical_feature=cat_cols)
cal = CalibratedClassifierCV(clf, method="isotonic", cv=3)
cal.fit(X_train, y_train)
cal_proba = cal.predict_proba(X_test)[:,1]


In [ ]:
import json
json.dump(list(X_train.columns), open("feature_order.json","w"))
json.dump(cat_cols, open("categorical_cols.json","w"))


In [ ]:
lgb.plot_importance(model, max_num_features=25, importance_type="gain", figsize=(8,6))


In [ ]:
import joblib, json
joblib.dump(model, "paid_claim_predictor_lightgbm.pkl")
json.dump({"features": list(X.columns),
           "categorical": cat_cols,
           "threshold": 0.5},
          open("model_config.json","w"))


In [ ]:
import joblib, pandas as pd
model = joblib.load("paid_claim_predictor_lightgbm.pkl")

def predict_claim(claim_row: pd.Series):
    proba = model.predict(claim_row.to_frame().T)[0]
    return int(proba >= 0.5), float(proba)


In [ ]:
import sys
# Publication: install dependencies from ../requirements.txt first.


In [ ]:
from fastapi import FastAPI
import joblib, pandas as pd
app = FastAPI()
model = joblib.load("paid_claim_predictor_lightgbm.pkl")

@app.post("/predict")
def predict(payload: dict):
    df = pd.DataFrame([payload])
    proba = model.predict(df)[0]
    return {"paid_probability": float(proba),
            "predicted_status": int(proba >= 0.5)}


In [ ]:
# Paths (adjust as needed)
MERGED_PATH = "cleaned_claims_dataset.csv"   # your final merged file
LINE_DATA_PATH = "claims_lines_dataset.csv"
MODEL_PATH = "paid_claim_predictor_lightgbm.pkl"
CONFIG_PATH = "model_config.json"
FEATURES_PATH = "feature_order.json"
CATEGORICALS_PATH = "categorical_cols.json"


In [ ]:
import pandas as pd
import numpy as np

# safer read to avoid mixed dtype issues
df = pd.read_csv(MERGED_PATH, low_memory=False)
print(df.shape)
df.head(3)


In [ ]:
# Convert all object-like columns to clean strings (uppercased, stripped) with "NONE" for missing
for c in df.columns:
    if df[c].dtype == 'object':
        df[c] = (df[c].astype('string')
                        .str.strip()
                        .str.upper()
                        .fillna("NONE"))

# Ensure target exists and is int 0/1
assert "paid_status" in df.columns, "Target 'paid_status' missing"
df["paid_status"] = df["paid_status"].astype(int)


In [ ]:
import pandas as pd, numpy as np, re, os

MERGED_PATH = "cleaned_claims_dataset.csv"
LINE_DATA_PATH = "claims_lines_dataset.csv"

def load_and_normalize_columns(path):
    df = pd.read_csv(path, low_memory=False)
    # normalize column names: strip spaces, collapse inner spaces, unify case style
    newcols = []
    for c in df.columns:
        cc = re.sub(r"\s+", "", c)  # remove all spaces
        newcols.append(cc)
    df.columns = newcols
    return df

df = load_and_normalize_columns(MERGED_PATH)
print("Loaded:", df.shape)
print("Example columns:", list(df.columns)[:30])

# Helpers
def normalize_paid_flag(v):
    if pd.isna(v): return np.nan
    s = str(v).strip().lower()
    if s in ("1","paid","yes","y","true"): return 1
    if s in ("0","notpaid","not paid","no","n","false","denied"): return 0
    # Heuristics: handle 'paid - CO:45', 'Paid\tCO:45', etc.
    if "paid" in s and "not" not in s:
        return 1
    if "not" in s or "denied" in s:
        return 0
    return np.nan

def find_label_col_for_slot(columns, k):
    """
    Try multiple common variants for slot label columns.
    Works on *normalized* column names (spaces removed).
    """
    candidates = [
        f"Paid{k}", f"PAID{k}", f"paid{k}",
        f"Paid{k}_Status", f"PAID{k}_STATUS", f"paid{k}_status",
        f"Paid{k}Status", f"PAID{k}STATUS", f"paid{k}status",
        f"Paid{k}StatusText", f"Paid{k}Flag"
    ]
    # also accept columns that *contain* 'Paid{k}' + 'Status' ignoring underscore/case
    regexes = [
        re.compile(fr"^paid{k}(_)?status$", re.I),
        re.compile(fr"^paid{k}$", re.I)
    ]
    # direct exact matches first
    for cand in candidates:
        if cand in columns:
            return cand
    # regex fallback
    for col in columns:
        for rx in regexes:
            if rx.match(col):
                return col
    return None

# Identify slot columns (procedures/modifiers) after normalization
proc_cols = [f"Proc{k}" for k in range(1,7) if f"Proc{k}" in df.columns]
mod_cols  = [f"Modifier{k}" for k in range(1,7) if f"Modifier{k}" in df.columns]

# Build a map of label columns per slot
label_map = {}
for k in range(1,7):
    lbl = find_label_col_for_slot(df.columns, k)
    label_map[k] = lbl

print("Per-slot label mapping found:")
for k in range(1,7):
    print(f"  slot {k}: {label_map[k]}")

# Figure out diag columns (normalized names like Diag1..Diag12)
diag_cols = [c for c in df.columns if re.fullmatch(r"Diag\d{1,2}", c)]
diag_cols = sorted(diag_cols, key=lambda x: int(re.findall(r"\d+", x)[0]))

# Fallback: claim-level target if present
claim_level_col = None
for cand in ["paid_status","PaidStatus","PAIDSTATUS"]:
    if cand in df.columns:
        claim_level_col = cand
        break
if claim_level_col:
    # normalize claim-level to 0/1
    df[claim_level_col] = df[claim_level_col].apply(normalize_paid_flag).astype("float")

rows = []
missing_label_count = 0
no_line_count = 0
total_lines_considered = 0

for _, r in df.iterrows():
    base = {
        "InsuID": r.get("InsuID", np.nan),
        "InsuPlanID": r.get("InsuPlanID", np.nan),
        "Diag1": r.get("Diag1", "NONE"),
        "Diag2": r.get("Diag2", "NONE"),
    }
    # count diags present
    n_diags_present = 0
    for dc in diag_cols:
        v = r.get(dc, None)
        if pd.notna(v) and str(v).strip():
            n_diags_present += 1
    base["n_diags_present"] = n_diags_present

    added_any = False
    for k in range(1,7):
        proc_col = f"Proc{k}"
        mod_col  = f"Modifier{k}"
        if proc_col not in df.columns:
            continue
        proc = r.get(proc_col, None)
        if pd.isna(proc) or str(proc).strip() in ("", "NONE"):
            continue

        total_lines_considered += 1

        # choose label column for this slot if available
        paid_col = label_map.get(k)
        raw_label = r.get(paid_col, np.nan) if paid_col else np.nan
        paid_line = normalize_paid_flag(raw_label)

        # fallback to claim-level if per-line label missing
        if pd.isna(paid_line) and claim_level_col:
            paid_line = r.get(claim_level_col, np.nan)

        if pd.isna(paid_line):
            missing_label_count += 1
            # If you prefer to SKIP unlabeled lines, continue (we do).
            continue

        rows.append({
            **base,
            "Proc": str(proc).strip().upper(),
            "Modifier": str(r.get(mod_col, "NONE")).strip().upper() if mod_col in df.columns else "NONE",
            "paid_line_status": int(paid_line)
        })
        added_any = True

    if not added_any:
        no_line_count += 1

line_df = pd.DataFrame(rows)

print("\n--- Build summary ---")
print("Total lines inspected      :", total_lines_considered)
print("Lines without usable label :", missing_label_count)
print("Claims with no usable line :", no_line_count)
print("line_df shape              :", line_df.shape)
print(line_df.head(5))

# Final cleanups
for c in ["InsuID","InsuPlanID","Proc","Modifier","Diag1","Diag2"]:
    if c in line_df.columns:
        line_df[c] = (line_df[c].astype("string").str.strip().str.upper().fillna("NONE"))

# Save if non-empty
if not line_df.empty:
    line_df.to_csv(LINE_DATA_PATH, index=False)
    vc = line_df["paid_line_status"].value_counts(dropna=False)
    print("\nSaved ->", LINE_DATA_PATH, "| class balance:", dict(vc))
else:
    print("\nline_df is empty — please scroll up and check:\n"
          "1) Per-slot mapping (slot k : column)\n"
          "2) Whether claim-level 'paid_status' exists\n"
          "3) That your merged file really contains Proc1..Proc6 and any paid flags")


In [ ]:
line_df.to_csv(LINE_DATA_PATH, index=False)


In [ ]:
line_df

In [ ]:
import pandas as pd
import numpy as np

LINE_DATA_PATH = "claims_lines_dataset.csv"
df = pd.read_csv(LINE_DATA_PATH, low_memory=False)
print("shape:", df.shape, "| unpaid %:", (df["paid_line_status"]==0).mean())

# Clean strings + NaNs
for c in ["InsuID","InsuPlanID","Proc","Modifier","Diag1","Diag2"]:
    df[c] = (df[c]
             .astype("string")
             .str.strip()
             .str.upper()
             .replace({"NAN":"NONE"})
             .fillna("NONE"))

# Keep target
y = df["paid_line_status"].astype(int)

# Base features
X = df[["InsuID","InsuPlanID","Proc","Modifier","Diag1","Diag2","n_diags_present"]].copy()
X["n_diags_present"] = pd.to_numeric(X["n_diags_present"], errors="coerce").fillna(0).astype(int)

# Quick peek
X.head(3)


In [ ]:
# Compact groupings
X["Proc_f2"] = X["Proc"].str[:2]
X["Proc_f3"] = X["Proc"].str[:3]
X["Diag1_ch"] = X["Diag1"].str[:1]   # ICD chapter (letter)
X["Diag1_c3"] = X["Diag1"].str[:3]   # ICD category
X["has_modifier"] = (X["Modifier"] != "NONE").astype(int)

# Categoricals
cat_cols = ["InsuID","InsuPlanID","Proc","Modifier","Diag1","Diag2","Proc_f2","Proc_f3","Diag1_ch","Diag1_c3"]
for c in cat_cols:
    X[c] = X[c].astype("category")


In [ ]:
from sklearn.model_selection import GroupKFold

def oof_frequency_encode(X_df, cols, groups, n_splits=5):
    gkf = GroupKFold(n_splits=n_splits)
    enc = {c: pd.Series(index=X_df.index, dtype=float) for c in cols}
    for tr_idx, va_idx in gkf.split(X_df, groups=groups):
        tr = X_df.iloc[tr_idx]
        for c in cols:
            freqs = tr[c].value_counts(dropna=False)
            enc[c].iloc[va_idx] = X_df[c].iloc[va_idx].map(freqs).fillna(0).astype(float)
    full_maps = {c: X_df[c].value_counts(dropna=False) for c in cols}
    return enc, full_maps

freq_cols = ["InsuPlanID","InsuID","Proc","Modifier","Diag1","Diag2","Proc_f3","Diag1_c3"]
groups = X["InsuPlanID"].astype(str)
n_splits = min(5, max(2, groups.nunique()))
enc, full_maps = oof_frequency_encode(X, freq_cols, groups, n_splits)

for c in freq_cols:
    X[f"{c}_freq"] = enc[c].astype(float)

# Class weights to emphasize unpaid (class 0)
n0, n1 = (y==0).sum(), (y==1).sum()
w0 = n1 / max(1, n0)     # ~ 21.9 in your case
import numpy as np
sample_weight = np.where(y==0, w0, 1.0)
print({"unpaid": int(n0), "paid": int(n1), "w0_unpaid": round(w0,2)})

# Final list of features (categoricals + numeric engineered)
cat_for_lgb = [c for c in X.columns if X[c].dtype.name=="category"]
features = list(X.columns)  # keep all we built; LightGBM will handle categories


In [ ]:
import lightgbm as lgb
from lightgbm import early_stopping, log_evaluation
from sklearn.metrics import roc_auc_score, precision_recall_fscore_support, accuracy_score
import numpy as np
import pandas as pd

params = {
    "objective": "binary",
    "metric": ["auc","binary_logloss"],
    "boosting_type": "gbdt",
    "num_leaves": 31,
    "min_data_in_leaf": 120,
    "lambda_l1": 0.2,
    "lambda_l2": 0.2,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "learning_rate": 0.05,
    "verbose": -1,
    "n_jobs": -1,
}

gkf = GroupKFold(n_splits=n_splits)
fold_rows, models = [], []

for fold, (tr, va) in enumerate(gkf.split(X, y, groups=groups), 1):
    X_tr, y_tr, w_tr = X.iloc[tr], y.iloc[tr], sample_weight[tr]
    X_va, y_va, w_va = X.iloc[va], y.iloc[va], sample_weight[va]

    dtr = lgb.Dataset(X_tr, label=y_tr, weight=w_tr, categorical_feature=cat_for_lgb, free_raw_data=False)
    dva = lgb.Dataset(X_va, label=y_va, weight=w_va, categorical_feature=cat_for_lgb, free_raw_data=False)

    model = lgb.train(
        params, dtr,
        valid_sets=[dtr, dva],
        valid_names=["train","valid"],
        num_boost_round=3000,
        callbacks=[early_stopping(100), log_evaluation(100)]
    )
    models.append(model)

    proba = model.predict(X_va, num_iteration=model.best_iteration)

    # pick threshold that maximizes class-0 F1
    best = None
    for t in np.linspace(0.1, 0.95, 36):
        pred = (proba >= t).astype(int)
        pr, rc, f1, _ = precision_recall_fscore_support(y_va, pred, average=None, labels=[0,1], zero_division=0)
        if best is None or f1[0] > best["f1_0"]:
            best = {"thr": float(t), "f1_0": float(f1[0]), "p0": float(pr[0]), "r0": float(rc[0])}

    pred05 = (proba >= 0.5).astype(int)
    pr05, rc05, f105, _ = precision_recall_fscore_support(y_va, pred05, average=None, labels=[0,1], zero_division=0)

    fold_rows.append({
        "fold": fold,
        "auc": roc_auc_score(y_va, proba),
        "acc@0.5": accuracy_score(y_va, pred05),
        "f1_0@0.5": f105[0], "p0@0.5": pr05[0], "r0@0.5": rc05[0],
        "best_thr": best["thr"], "f1_0@best": best["f1_0"], "p0@best": best["p0"], "r0@best": best["r0"],
        "best_iter": model.best_iteration
    })

cv_report = pd.DataFrame(fold_rows)
cv_report


In [ ]:
import joblib, json, os

best_iter = int(np.mean([m.best_iteration for m in models]))
final_ds = lgb.Dataset(X, label=y, weight=sample_weight, categorical_feature=cat_for_lgb, free_raw_data=False)
final_model = lgb.train(params, final_ds, num_boost_round=best_iter)

deploy_thr = float(np.median(cv_report["best_thr"]))
print("Deploy threshold:", deploy_thr, "| best_iter:", best_iter)

# Save artifacts
joblib.dump(final_model, "paid_line_lightgbm_v2.pkl")
json.dump(features, open("feature_list_v2.json","w"))
json.dump(cat_for_lgb, open("categorical_cols_v2.json","w"))
json.dump({"threshold": deploy_thr, "best_iter": best_iter}, open("config_v2.json","w"))

# Save frequency maps for production use (recreate freq features on new data)
import pathlib
pathlib.Path("freqmaps_v2").mkdir(exist_ok=True)
for c, mp in full_maps.items():
    mp.to_json(f"freqmaps_v2/{c}_freq.json")


In [ ]:
import json, joblib, pandas as pd

model = joblib.load("paid_line_lightgbm_v2.pkl")
features = json.load(open("feature_list_v2.json"))
cat_for_lgb = json.load(open("categorical_cols_v2.json"))
cfg = json.load(open("config_v2.json"))
deploy_thr = cfg["threshold"]

# load freq maps
import os
freq_maps = {}
for fn in os.listdir("freqmaps_v2"):
    if fn.endswith("_freq.json"):
        base = fn.replace("_freq.json","")
        # Fix: Load JSON as dictionary first, then convert to DataFrame with proper structure
        freq_data = json.load(open(os.path.join("freqmaps_v2", fn)))
        # Create DataFrame with the frequency data as Series
        freq_maps[base] = pd.DataFrame(list(freq_data.items()), columns=[base, 'frequency']).set_index(base)

def add_features(df_in):
    df = df_in.copy()
    for c in ["InsuID","InsuPlanID","Proc","Modifier","Diag1","Diag2"]:
        df[c] = df[c].astype("string").str.strip().str.upper().fillna("NONE")
    if "n_diags_present" not in df: df["n_diags_present"] = 0
    df["Proc_f2"] = df["Proc"].str[:2]
    df["Proc_f3"] = df["Proc"].str[:3]
    df["Diag1_ch"] = df["Diag1"].str[:1]
    df["Diag1_c3"] = df["Diag1"].str[:3]
    df["has_modifier"] = (df["Modifier"] != "NONE").astype(int)
    # freq features
    for base in ["InsuPlanID","InsuID","Proc","Modifier","Diag1","Diag2","Proc_f3","Diag1_c3"]:
        mp = freq_maps.get(base)
        if mp is not None:
            # Fix: Use the proper column name for mapping
            df[f"{base}_freq"] = df[base].map(mp['frequency']).fillna(0).astype(float)
        else:
            df[f"{base}_freq"] = 0.0
    for c in features:
        if c in cat_for_lgb:
            df[c] = df[c].astype("category")
    return df[features]

def predict_line(payload: dict, thr=None):
    thr = deploy_thr if thr is None else float(thr)
    dfp = add_features(pd.DataFrame([payload]))
    proba = float(model.predict(dfp)[0])
    return {"paid_probability": proba, "predicted_status": int(proba >= thr), "threshold": thr}

# test
predict_line({
    "InsuID": "109","InsuPlanID":"41","Proc":"97110","Modifier":"GP","Diag1":"M54.2","Diag2":"NONE","n_diags_present":1
})

In [ ]:
import pandas as pd

df = pd.read_csv(MERGED_PATH, low_memory=False)
print("Cols:", len(df.columns))
print(sorted(df.columns)[:80])  # peek

# See which label name patterns exist for each slot
for k in range(1,7):
    candidates = [f"Paid{k}", f"Paid{k}_Status", f"PAID{k}", f"PAID{k}_STATUS", f"Paid{k}Status"]
    present = [c for c in candidates if c in df.columns]
    print(f"k={k}: present label columns -> {present}")


In [ ]:
import numpy as np

def normalize_paid_flag(v):
    if pd.isna(v): 
        return np.nan
    s = str(v).strip().lower()
    if s in ("1","paid","yes","y","true","co:45 paid","co45 paid"):  # add any synonyms you observe
        return 1
    if s in ("0","not paid","no","n","false","denied"):
        return 0
    # Sometimes it's 'Paid\tCO:45' or 'Paid - CO:45', try a heuristic
    if "paid" in s and "not" not in s:
        return 1
    if "not paid" in s or "denied" in s:
        return 0
    return np.nan

slot_cols = [(f"Proc{k}", f"Modifier{k}") for k in range(1,7)]
diag_cols = [c for c in df.columns if c.upper().startswith("DIAG")]

# Map which label col to use per slot (first match wins)
label_map = {}
for k in range(1,7):
    candidates = [f"Paid{k}", f"Paid{k}_Status", f"PAID{k}", f"PAID{k}_STATUS", f"Paid{k}Status"]
    label_map[k] = next((c for c in candidates if c in df.columns), None)

print("Label map (slot -> column):", label_map)

rows = []
missing_label_count = 0
no_proc_count = 0

for _, r in df.iterrows():
    base = {
        "InsuID": r.get("InsuID"),
        "InsuPlanID": r.get("InsuPlanID"),
        "Diag1": r.get("Diag1", "NONE"),
        "Diag2": r.get("Diag2", "NONE"),
    }
    # basic context
    n_diags_present = 0
    for dc in diag_cols:
        v = r.get(dc)
        if pd.notna(v) and str(v).strip():
            n_diags_present += 1
    base["n_diags_present"] = n_diags_present

    added_any = False
    for k, (proc_col, mod_col) in enumerate(slot_cols, start=1):
        proc = r.get(proc_col)
        if pd.isna(proc) or str(proc).strip() == "":
            continue

        # choose the right label column for this slot
        paid_col = label_map.get(k)
        paid_val_raw = r.get(paid_col) if paid_col else np.nan
        paid_line_status = normalize_paid_flag(paid_val_raw)

        if pd.isna(paid_line_status):
            missing_label_count += 1
            # If you want to fallback to claim-level paid_status (weaker, but avoids dropping everything):
            if "paid_status" in r.index and pd.notna(r["paid_status"]):
                paid_line_status = int(r["paid_status"])
            else:
                # skip this line if we truly don't have a label
                continue

        rows.append({
            **base,
            "Proc": str(proc).strip().upper(),
            "Modifier": str(r.get(mod_col, "NONE")).strip().upper() if pd.notna(r.get(mod_col)) else "NONE",
            "paid_line_status": int(paid_line_status)
        })
        added_any = True

    if not added_any:
        no_proc_count += 1

line_df = pd.DataFrame(rows)
print("Line-level shape:", line_df.shape)
print("Lines with fallback labels used:", missing_label_count)
print("Claims with no usable procs:", no_proc_count)

# Clean strings
for c in ["InsuID","InsuPlanID","Proc","Modifier","Diag1","Diag2"]:
    line_df[c] = (line_df[c].astype('string').str.strip().str.upper().fillna("NONE"))

line_df["paid_line_status"] = line_df["paid_line_status"].astype(int)
line_df.head()


In [ ]:
from sklearn.model_selection import GroupKFold
import numpy as np

use = line_df.copy()
X = use[["InsuID","InsuPlanID","Proc","Modifier","Diag1","Diag2","n_diags_present"]]
y = use["paid_line_status"].astype(int)

# Categorical handling
cat_cols = ["InsuID","InsuPlanID","Proc","Modifier","Diag1","Diag2"]
for c in cat_cols:
    X[c] = X[c].astype("category")

groups = X["InsuPlanID"].astype(str)
n_samples = len(X)
n_unique_groups = groups.nunique()
n_splits = int(min(5, max(2, n_unique_groups)))  # at least 2, at most 5, and not more than unique groups

print(f"Samples={n_samples}, unique_groups={n_unique_groups}, using n_splits={n_splits}")
assert n_samples > 0, "No samples to split — check line_df creation step."


In [ ]:
import lightgbm as lgb
from lightgbm import early_stopping, log_evaluation
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score, accuracy_score

gkf = GroupKFold(n_splits=n_splits)
fold_metrics, models = [], []

params = {
    "objective": "binary",
    "metric": ["auc","binary_logloss"],
    "boosting_type": "gbdt",
    "num_leaves": 63,
    "learning_rate": 0.05,
    "feature_fraction": 0.9,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "verbose": -1,
    "n_jobs": -1,
}

for fold, (tr, va) in enumerate(gkf.split(X, y, groups=groups), 1):
    X_tr, X_va = X.iloc[tr], X.iloc[va]
    y_tr, y_va = y.iloc[tr], y.iloc[va]

    ds_tr = lgb.Dataset(X_tr, label=y_tr, categorical_feature=cat_cols, free_raw_data=False)
    ds_va = lgb.Dataset(X_va, label=y_va, categorical_feature=cat_cols, free_raw_data=False)

    model = lgb.train(
        params,
        ds_tr,
        valid_sets=[ds_tr, ds_va],
        valid_names=["train","valid"],
        num_boost_round=3000,
        callbacks=[
            early_stopping(stopping_rounds=100),
            log_evaluation(period=100),
        ],
    )
    models.append(model)

    proba = model.predict(X_va, num_iteration=model.best_iteration)
    pred = (proba >= 0.5).astype(int)

    metrics = {
        "fold": fold,
        "auc": roc_auc_score(y_va, proba),
        "accuracy": accuracy_score(y_va, pred),
        "f1": f1_score(y_va, pred),
        "precision": precision_score(y_va, pred),
        "recall": recall_score(y_va, pred),
        "best_iter": model.best_iteration
    }
    fold_metrics.append(metrics)

pd.DataFrame(fold_metrics)


In [ ]:
import matplotlib.pyplot as plt
import lightgbm as lgb

# Plot feature importances
lgb.plot_importance(models[0], max_num_features=20)
plt.show()


In [ ]:
best_iter = int(np.mean([m.best_iteration for m in models]))
final_ds = lgb.Dataset(X, label=y, categorical_feature=cat_cols)
final_model = lgb.train(params, final_ds, num_boost_round=best_iter)


In [ ]:
import joblib, json

joblib.dump(final_model, "paid_claims_model.pkl")
json.dump(list(X.columns), open("feature_list.json","w"))
json.dump(cat_cols, open("categorical_cols.json","w"))
json.dump({"threshold":0.5, "best_iter":best_iter}, open("config.json","w"))


In [ ]:
import pandas as pd, joblib, json
model = joblib.load("paid_claims_model.pkl")
features = json.load(open("feature_list.json"))
cats = json.load(open("categorical_cols.json"))
cfg = json.load(open("config.json"))

def predict_claim(line_dict):
    df = pd.DataFrame([line_dict], columns=features)
    for c in cats: df[c] = df[c].astype("category")
    proba = model.predict(df)[0]
    return {"probability": float(proba),
            "predicted_status": int(proba >= cfg["threshold"])}


In [ ]:
def preprocess_payload(payload: dict) -> pd.DataFrame:
    """
    payload -> single-row DataFrame with the exact feature order and dtypes used in training.
    """
    import pandas as pd

    # Build DF with correct column order
    df = pd.DataFrame([payload], columns=features)

    # String cleanup + uppercase for categorical columns
    for c in features:
        if c in cat_cols:
            df[c] = (
                df[c]
                .astype("string")
                .str.strip()
                .str.upper()
                .fillna("NONE")
                .astype("category")
            )
        else:
            # numeric-like feature (e.g., counts)
            df[c] = pd.to_numeric(df[c], errors="coerce")

    # Basic fill for numerics if needed
    df = df.fillna({"n_diags_present": 0})
    return df


In [ ]:
threshold = cfg.get("threshold", 0.5)

In [ ]:
import pandas as pd

# If you saved these earlier; otherwise keep using in-memory `predict_line`, `deploy_thr`
# from your notebook cells:
# from my_inference_module import predict_line, deploy_thr

tests = [
    {
        "name": "PT visit 97110 with GP (typical paid)",
        "payload": {
            "InsuID": "109","InsuPlanID":"41",
            "Proc":"97110","Modifier":"GP",
            "Diag1":"M54.2","Diag2":"NONE","n_diags_present":1
        },
        "expected": 1
    },
    {
        "name": "Echocardiography 93306 (auth-sensitive, assume no auth)",
        "payload": {
            "InsuID":"47","InsuPlanID":"41",
            "Proc":"93306","Modifier":"NONE",
            "Diag1":"R07.9","Diag2":"R00.2","n_diags_present":2
        },
        "expected": 0
    },
    {
        "name": "DME frames V2020 (documentation sensitive)",
        "payload": {
            "InsuID":"NY1652","InsuPlanID":"H35",
            "Proc":"V2020","Modifier":"NONE",
            "Diag1":"H52.4","Diag2":"NONE","n_diags_present":1
        },
        "expected": 0
    },
    {
        "name": "Chronic care mgmt 99457 (typical paid)",
        "payload": {
            "InsuID":"109","InsuPlanID":"E03",
            "Proc":"99457","Modifier":"NONE",
            "Diag1":"I10","Diag2":"E03.9","n_diags_present":2
        },
        "expected": 1
    },
    {
        "name": "Fundus photos 92250 (coverage tricky; assume unpaid)",
        "payload": {
            "InsuID":"NY1652","InsuPlanID":"H35",
            "Proc":"92250","Modifier":"NONE",
            "Diag1":"H35.013","Diag2":"H52.4","n_diags_present":2
        },
        "expected": 0
    }
]

def run_synthetic_tests(thr=None):
    thr = deploy_thr if thr is None else float(thr)
    rows = []
    y_true, y_prob, y_pred = [], [], []
    for t in tests:
        res = predict_line(t["payload"], thr)
        rows.append({
            "name": t["name"],
            "expected": t["expected"],
            "prob": round(res["paid_probability"], 4),
            "pred": res["predicted_status"],
            "thr": round(res["threshold"], 4),
            "correct": int(res["predicted_status"] == t["expected"])
        })
        y_true.append(t["expected"])
        y_prob.append(res["paid_probability"])
        y_pred.append(res["predicted_status"])
    df_out = pd.DataFrame(rows)
    return df_out, pd.Series(y_true, name="y_true"), pd.Series(y_prob, name="y_prob"), pd.Series(y_pred, name="y_pred")

df_syn, y_true, y_prob, y_pred = run_synthetic_tests()
df_syn


In [ ]:
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_recall_fscore_support, confusion_matrix
)
import numpy as np

acc = accuracy_score(y_true, y_pred)
# AUC needs both classes present; handle edge case:
try:
    auc = roc_auc_score(y_true, y_prob)
except ValueError:
    auc = float("nan")

# Class-wise P/R/F1 (labels=[0,1] ensures class-0 shown first)
p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average=None, labels=[0,1], zero_division=0)
cm = confusion_matrix(y_true, y_pred, labels=[0,1])

print("Synthetic 5-case results")
print(df_syn)
print("\nConfusion Matrix [rows=true 0/1, cols=pred 0/1]:\n", cm)
print(f"\nAccuracy : {acc:.3f}")
print(f"AUC      : {auc:.3f}")
print(f"class-0  : P={p[0]:.3f} R={r[0]:.3f} F1={f1[0]:.3f}")
print(f"class-1  : P={p[1]:.3f} R={r[1]:.3f} F1={f1[1]:.3f}")

# Compare to production targets
targets = {'Recall0': 0.95, 'Precision0': 0.80, 'AUC': 0.98}
print("\nTargets (production-grade):", targets)
